# Normalize `extracted_features` into lookup tables

Turns the flat LLM output table into a star-like layout: one `*_lookup` table per categorical dimension (integer id ↔ value), a `company_lookup` table, and `company_feature_keys`, which stores every company as integer foreign keys ready for ML.


In [2]:
import pymysql
import pandas as pd

import os
from dotenv import load_dotenv
load_dotenv()

conn = pymysql.connect(
    host=os.getenv("MYSQL_HOST", "localhost"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    database=os.getenv("MYSQL_DATABASE", "db_all"),
    charset="utf8mb4"
)

query = "SELECT * FROM extracted_features LIMIT 10;"

df = pd.read_sql(query, conn)

df

,company_name,industry,core_technology,product_type,target_audience,monetization_strategy,delivery_medium,value_proposition,regulatory_environment,unified_text,error,label
0,&AI,other_unknown,other_unknown,software_platform,enterprises,subscription,web_application,workflow_automation,other_unknown,Company: &AI. One-liner: Collaborative workspa...,None,1
1,10 By 10,other_unknown,ai_ml,two_sided_marketplace,enterprises,transaction_fee,web_application,cost_efficiency,data_privacy,Company: 10 By 10. One-liner: 10by10 is a uniq...,None,1
2,1000Memories,other_unknown,other_unknown,software_platform,individual_consumers,other_unknown,web_application,other_unknown,unregulated_or_low_friction,Company: 1000Memories. One-liner: Memorial sit...,None,1
3,100x,enterprise_software,ai_ml,software_platform,enterprises,other_unknown,web_application,workflow_automation,unregulated_or_low_friction,"Company: 100x. One-liner: Create, Share, and M...",None,1
4,10x,healthcare_and_biotech,ai_ml,managed_service,small_and_medium_businesses,subscription,api_integration,workflow_automation,healthcare_compliance,Company: 10x. One-liner: AI Sales Teams for De...,None,1
5,10x Science,healthcare_and_biotech,ai_ml,software_platform,enterprises,subscription,web_application,workflow_automation,healthcare_compliance,Company: 10x Science. One-liner: The AI-native...,None,1
6,14.ai,e_commerce_and_retail,ai_ml,software_platform,enterprises,other_unknown,web_application,workflow_automation,unregulated_or_low_friction,Company: 14.ai. One-liner: AI engine powering ...,None,1
7,1849 bio,industrials_and_manufacturing,other_unknown,other_unknown,enterprises,direct_sales_or_one_time,other_unknown,cost_efficiency,unregulated_or_low_friction,Company: 1849 bio. One-liner: Engineering micr...,None,1
8,1910,healthcare_and_biotech,ai_ml,software_platform,enterprises,other_unknown,other_unknown,workflow_automation,healthcare_compliance,Company: 1910. One-liner: Multimodal AI Platfo...,None,1
9,1stCollab,e_commerce_and_retail,ai_ml,software_platform,enterprises,subscription,web_application,workflow_automation,unregulated_or_low_friction,Company: 1stCollab. One-liner: Performance-Opt...,None,1


In [95]:
import pandas as pd

database_name = "db_all"
tables = pd.read_sql(f"SHOW TABLES FROM {database_name};", conn)
tables

,Tables_in_db_all
0,company_core_technology
1,company_delivery_medium
2,company_industry
3,company_label
4,company_lookup
5,company_monetization_strategy
6,company_product_type
7,company_regulatory_environment
8,company_target_audience
9,company_unified_text


In [59]:
pd.read_sql("DESCRIBE extracted_features;", conn)

,Field,Type,Null,Key,Default,Extra
0,company_name,varchar(255),NO,PRI,None,
1,industry,text,YES,,None,
2,core_technology,text,YES,,None,
3,product_type,text,YES,,None,
4,target_audience,text,YES,,None,
5,monetization_strategy,text,YES,,None,
6,delivery_medium,text,YES,,None,
7,value_proposition,text,YES,,None,
8,regulatory_environment,text,YES,,None,
9,unified_text,text,YES,,None,


In [60]:
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS company_lookup;")

cursor.execute("""
CREATE TABLE company_lookup AS
SELECT
    ROW_NUMBER() OVER (ORDER BY company_name) - 1 AS company_id,
    company_name
FROM extracted_features;
""")

cursor.execute("""
ALTER TABLE company_lookup
ADD PRIMARY KEY (company_id);
""")

conn.commit()

In [73]:
cursor = conn.cursor()

cursor.execute("""
ALTER TABLE company_lookup
ADD COLUMN error TEXT;
""")

cursor.execute("""
UPDATE company_lookup cl
JOIN extracted_features ef
    ON cl.company_name = ef.company_name
SET cl.error = ef.error;
""")

conn.commit()

In [74]:
pd.read_sql("SELECT * FROM company_lookup LIMIT 20;", conn)

,company_id,company_name,error
0,0,&AI,None
1,1,10 By 10,None
2,2,1000Memories,None
3,3,100x,None
4,4,10x,None
5,5,10x Science,None
6,6,14.ai,None
7,7,1849 bio,None
8,8,1910,None
9,9,1stCollab,None


In [61]:
pd.read_sql("SELECT * FROM company_lookup LIMIT 20;", conn)

,company_id,company_name
0,0,&AI
1,1,10 By 10
2,2,1000Memories
3,3,100x
4,4,10x
5,5,10x Science
6,6,14.ai
7,7,1849 bio
8,8,1910
9,9,1stCollab


In [68]:
cursor = conn.cursor()

lookup_columns = [
    "industry",
    "core_technology",
    "product_type",
    "target_audience",
    "monetization_strategy",
    "delivery_medium",
    "value_proposition",
    "regulatory_environment",
    "unified_text",
    "label"
]

for col in lookup_columns:
    table_name = f"{col}_lookup"
    id_name = f"{col}_id"

    cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`;")

    cursor.execute(f"""
    CREATE TABLE `{table_name}` AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY `{col}`) - 1 AS `{id_name}`,
        `{col}`
    FROM (
        SELECT DISTINCT `{col}`
        FROM extracted_features
        WHERE `{col}` IS NOT NULL
    ) AS unique_values;
    """)

    cursor.execute(f"""
    ALTER TABLE `{table_name}`
    ADD PRIMARY KEY (`{id_name}`);
    """)

conn.commit()

In [96]:
pd.read_sql("SHOW TABLES;", conn)

,Tables_in_db_all
0,company_core_technology
1,company_delivery_medium
2,company_industry
3,company_label
4,company_lookup
5,company_monetization_strategy
6,company_product_type
7,company_regulatory_environment
8,company_target_audience
9,company_unified_text


In [64]:
pd.read_sql("SELECT * FROM industry_lookup LIMIT 20;", conn)

,industry_id,industry
0,0,e_commerce_and_retail
1,1,education
2,2,enterprise_software
3,3,fintech
4,4,healthcare_and_biotech
5,5,industrials_and_manufacturing
6,6,other_unknown
7,7,real_estate_and_proptech


In [108]:
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS company_feature_keys;")

cursor.execute("""
CREATE TABLE company_feature_keys AS
SELECT
    cl.company_id,
    il.industry_id,
    ctl.core_technology_id,
    ptl.product_type_id,
    tal.target_audience_id,
    msl.monetization_strategy_id,
    dml.delivery_medium_id,
    vpl.value_proposition_id,
    rel.regulatory_environment_id,
    ll.label_id
FROM extracted_features ef

JOIN company_lookup cl
    ON ef.company_name = cl.company_name

LEFT JOIN industry_lookup il
    ON ef.industry = il.industry

LEFT JOIN core_technology_lookup ctl
    ON ef.core_technology = ctl.core_technology

LEFT JOIN product_type_lookup ptl
    ON ef.product_type = ptl.product_type

LEFT JOIN target_audience_lookup tal
    ON ef.target_audience = tal.target_audience

LEFT JOIN monetization_strategy_lookup msl
    ON ef.monetization_strategy = msl.monetization_strategy

LEFT JOIN delivery_medium_lookup dml
    ON ef.delivery_medium = dml.delivery_medium

LEFT JOIN value_proposition_lookup vpl
    ON ef.value_proposition = vpl.value_proposition

LEFT JOIN regulatory_environment_lookup rel
    ON ef.regulatory_environment = rel.regulatory_environment

LEFT JOIN label_lookup ll
    ON ef.label = ll.label;
""")

cursor.execute("""
ALTER TABLE company_feature_keys
ADD PRIMARY KEY (company_id);
""")

conn.commit()

In [109]:
pd.read_sql("SELECT * FROM company_feature_keys LIMIT 20;", conn)

,company_id,industry_id,core_technology_id,product_type_id,target_audience_id,monetization_strategy_id,delivery_medium_id,value_proposition_id,regulatory_environment_id,label_id
0,0,6,5,2,1,5,6,5,3,1
1,1,6,0,4,1,6,6,1,0,1
2,2,6,5,2,2,4,6,2,4,1
3,3,2,0,2,1,4,6,5,4,1
4,4,4,0,0,4,5,0,5,2,1
5,5,4,0,2,1,5,6,5,2,1
6,6,0,0,2,1,4,6,5,4,1
7,7,5,5,1,1,1,5,1,4,1
8,8,4,0,2,1,4,5,5,2,1
9,9,0,0,2,1,5,6,5,4,1


In [111]:
cursor = conn.cursor()

# Add the new columns
cursor.execute("""
ALTER TABLE company_feature_keys
ADD COLUMN company_name VARCHAR(255),
ADD COLUMN unified_text TEXT,
ADD COLUMN error TEXT;
""")

# Fill them using extracted_features + company_lookup
cursor.execute("""
UPDATE company_feature_keys cfk
JOIN company_lookup cl
    ON cfk.company_id = cl.company_id
JOIN extracted_features ef
    ON cl.company_name = ef.company_name
SET
    cfk.company_name = ef.company_name,
    cfk.unified_text = ef.unified_text,
    cfk.error = ef.error;
""")

conn.commit()

In [112]:
pd.read_sql("SELECT * FROM company_feature_keys LIMIT 10;", conn)

,company_id,industry_id,core_technology_id,product_type_id,target_audience_id,monetization_strategy_id,delivery_medium_id,value_proposition_id,regulatory_environment_id,label_id,company_name,unified_text,error
0,0,6,5,2,1,5,6,5,3,1,&AI,Company: &AI. One-liner: Collaborative workspa...,None
1,1,6,0,4,1,6,6,1,0,1,10 By 10,Company: 10 By 10. One-liner: 10by10 is a uniq...,None
2,2,6,5,2,2,4,6,2,4,1,1000Memories,Company: 1000Memories. One-liner: Memorial sit...,None
3,3,2,0,2,1,4,6,5,4,1,100x,"Company: 100x. One-liner: Create, Share, and M...",None
4,4,4,0,0,4,5,0,5,2,1,10x,Company: 10x. One-liner: AI Sales Teams for De...,None
5,5,4,0,2,1,5,6,5,2,1,10x Science,Company: 10x Science. One-liner: The AI-native...,None
6,6,0,0,2,1,4,6,5,4,1,14.ai,Company: 14.ai. One-liner: AI engine powering ...,None
7,7,5,5,1,1,1,5,1,4,1,1849 bio,Company: 1849 bio. One-liner: Engineering micr...,None
8,8,4,0,2,1,4,5,5,2,1,1910,Company: 1910. One-liner: Multimodal AI Platfo...,None
9,9,0,0,2,1,5,6,5,4,1,1stCollab,Company: 1stCollab. One-liner: Performance-Opt...,None


In [113]:
cursor = conn.cursor()

cursor.execute("""
ALTER TABLE company_feature_keys
MODIFY COLUMN company_name VARCHAR(255) AFTER company_id;
""")

conn.commit()

In [114]:
pd.read_sql("DESCRIBE company_feature_keys;", conn)

,Field,Type,Null,Key,Default,Extra
0,company_id,bigint unsigned,NO,PRI,0,
1,company_name,varchar(255),YES,,None,
2,industry_id,bigint unsigned,YES,,0,
3,core_technology_id,bigint unsigned,YES,,0,
4,product_type_id,bigint unsigned,YES,,0,
5,target_audience_id,bigint unsigned,YES,,0,
6,monetization_strategy_id,bigint unsigned,YES,,0,
7,delivery_medium_id,bigint unsigned,YES,,0,
8,value_proposition_id,bigint unsigned,YES,,0,
9,regulatory_environment_id,bigint unsigned,YES,,0,


In [115]:
pd.read_sql("SELECT * FROM company_feature_keys LIMIT 10;", conn)

,company_id,company_name,industry_id,core_technology_id,product_type_id,target_audience_id,monetization_strategy_id,delivery_medium_id,value_proposition_id,regulatory_environment_id,label_id,unified_text,error
0,0,&AI,6,5,2,1,5,6,5,3,1,Company: &AI. One-liner: Collaborative workspa...,None
1,1,10 By 10,6,0,4,1,6,6,1,0,1,Company: 10 By 10. One-liner: 10by10 is a uniq...,None
2,2,1000Memories,6,5,2,2,4,6,2,4,1,Company: 1000Memories. One-liner: Memorial sit...,None
3,3,100x,2,0,2,1,4,6,5,4,1,"Company: 100x. One-liner: Create, Share, and M...",None
4,4,10x,4,0,0,4,5,0,5,2,1,Company: 10x. One-liner: AI Sales Teams for De...,None
5,5,10x Science,4,0,2,1,5,6,5,2,1,Company: 10x Science. One-liner: The AI-native...,None
6,6,14.ai,0,0,2,1,4,6,5,4,1,Company: 14.ai. One-liner: AI engine powering ...,None
7,7,1849 bio,5,5,1,1,1,5,1,4,1,Company: 1849 bio. One-liner: Engineering micr...,None
8,8,1910,4,0,2,1,4,5,5,2,1,Company: 1910. One-liner: Multimodal AI Platfo...,None
9,9,1stCollab,0,0,2,1,5,6,5,4,1,Company: 1stCollab. One-liner: Performance-Opt...,None
